In [4]:
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from collections import defaultdict
import pandas as pd

###############################################################################
# 1) Define Arguments
###############################################################################
args = {
    "models": ["nlpaueb/sec-bert-base", "bert-base-uncased"],
    "datasets": [
        {
            "name": "yelp_review_full",
            "config": None,
            "split": "train",
            "text_column": "text",
        },
        {
            "name": "wikitext",
            "config": "wikitext-2-raw-v1",
            "split": "train",
            "text_column": "text",
        },
        {"name": "ag_news", "config": None, "split": "train", "text_column": "text"},
    ],
    "max_texts": 1000,
    "batch_size": 64,
    "num_drift_levels": 5,
    "drift_strengths": [0.0, 0.5, 1.0],
    "pca_components": 2,
    "output_dir": "results_multidataset",
}

os.makedirs(args["output_dir"], exist_ok=True)

###############################################################################
# 2) Utility Functions
###############################################################################


def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]


def extract_cls_embeddings(model, tokenizer, texts, device):
    encodings = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=128
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()


def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            new_texts.append(txt)
            continue
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]
        new_texts.append(" ".join(words))
    return new_texts


###############################################################################
# 3) DriftDetector Class
###############################################################################
class DriftDetector:
    def __init__(
        self, model, tokenizer, device, batch_generator, args, pca_transform=None
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.batch_generator = batch_generator
        self.args = args
        self.pca_transform = pca_transform
        self.prototype = None
        self.prototypes = []
        self.cosine_scores = []

    def initialize_baseline(self, texts):
        embeddings = []
        for batch in self.batch_generator(texts, self.args["batch_size"]):
            cls_emb = extract_cls_embeddings(
                self.model, self.tokenizer, batch, self.device
            )
            embeddings.append(cls_emb)
        all_embeddings = np.concatenate(embeddings, axis=0)
        if self.pca_transform is not None:
            all_embeddings = self.pca_transform.transform(all_embeddings)
        self.prototype = np.mean(all_embeddings, axis=0)
        self.prototypes.append(self.prototype)

    def detect_drifts(self, texts):
        for batch in tqdm(self.batch_generator(texts, self.args["batch_size"])):
            batch_embeddings = extract_cls_embeddings(
                self.model, self.tokenizer, batch, self.device
            )
            if self.pca_transform is not None:
                batch_embeddings = self.pca_transform.transform(batch_embeddings)
            mean_emb = batch_embeddings.mean(axis=0, keepdims=True)
            sim = cosine_similarity(mean_emb, [self.prototype])[0][0]
            self.cosine_scores.append(sim)
            self._update_prototype(batch_embeddings)

    def _update_prototype(self, batch_embeddings):
        delta = batch_embeddings - self.prototype
        distances = np.linalg.norm(delta, axis=1)
        weights = np.exp(-distances / 2.0)
        weighted_sum = np.sum(weights[:, None] * delta, axis=0)
        self.prototype += weighted_sum / np.sum(weights)
        self.prototypes.append(self.prototype)


###############################################################################
# 4) Main Flow
###############################################################################
def main():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    os.makedirs(args["output_dir"], exist_ok=True)
    results = []

    for dataset_info in args["datasets"]:
        dataset_name = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col = dataset_info["text_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = ds[text_col]
        texts = list(texts)
        random.shuffle(texts)
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]

        half_point = len(texts) // 2
        baseline_texts = texts[:half_point]

        for model_name in args["models"]:
            print(f"\n--- Using Model: {model_name} ---")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name)
            model.to(device)
            model.eval()

            for drift_strength in args["drift_strengths"]:
                print(f"\nSimulating drift with strength={drift_strength}")
                drifted_texts = introduce_gradual_drift(
                    texts[half_point:], fraction_shuffle=drift_strength
                )
                test_texts = baseline_texts + drifted_texts

                detector_no_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=None,
                )
                detector_no_pca.initialize_baseline(baseline_texts)
                detector_no_pca.detect_drifts(test_texts)

                baseline_embs = []
                for b in batch_generator(baseline_texts, args["batch_size"]):
                    emb_b = extract_cls_embeddings(model, tokenizer, b, device)
                    baseline_embs.append(emb_b)
                baseline_embs = np.concatenate(baseline_embs, axis=0)

                pca = PCA(n_components=args["pca_components"])
                pca.fit(baseline_embs)
                baseline_embs_2d = pca.transform(baseline_embs)

                drifted_embs_full = extract_cls_embeddings(
                    model, tokenizer, drifted_texts, device
                )
                drifted_embs_2d = pca.transform(drifted_embs_full)

                detector_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=pca,
                )
                detector_pca.initialize_baseline(baseline_texts)
                detector_pca.detect_drifts(test_texts)

                results.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "drift_strength": drift_strength,
                        "pca": False,
                        "cosine_scores": detector_no_pca.cosine_scores,
                    }
                )

                results.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "drift_strength": drift_strength,
                        "pca": True,
                        "cosine_scores": detector_pca.cosine_scores,
                        "baseline_embs": baseline_embs_2d,
                        "drifted_embs": drifted_embs_2d,
                    }
                )

                print(
                    f"  Cosine scores (no PCA): {len(detector_no_pca.cosine_scores)} entries"
                )
                print(
                    f"  Cosine scores (PCA): {len(detector_pca.cosine_scores)} entries"
                )

    print("\n=== Plotting results for each (dataset, model, drift_strength) ===")
    grouped = defaultdict(list)
    for r in results:
        key = (r["dataset"], r["model"], r["drift_strength"])
        grouped[key].append(r)

    for (dataset_name, model_name, drift_strength), group_vals in grouped.items():
        fig, axs = plt.subplots(2, 3, figsize=(18, 10))

        ax = axs[0, 0]
        for gv in group_vals:
            label_suffix = "PCA" if gv["pca"] else "No PCA"
            ax.plot(gv["cosine_scores"], label=f"{label_suffix}")
        drift_start = len(group_vals[0]["cosine_scores"]) // 2
        ax.axvline(x=drift_start, color="red", linestyle="--", label="Drift Start")
        ax.set_title("Cosine Similarity vs Batch Index")
        ax.set_xlabel("Batch Index")
        ax.set_ylabel("Cosine Similarity")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)

        ax = axs[0, 1]
        for gv in group_vals:
            cosine_series = pd.Series(gv["cosine_scores"])
            rolling_mean = cosine_series.rolling(window=3).mean()
            rolling_std = cosine_series.rolling(window=3).std()
            label_suffix = "PCA" if gv["pca"] else "No PCA"
            ax.plot(rolling_mean, label=f"{label_suffix} - Rolling Mean")
            ax.fill_between(
                range(len(rolling_mean)),
                rolling_mean - rolling_std,
                rolling_mean + rolling_std,
                alpha=0.2,
                label=f"{label_suffix} - Rolling Std",
            )
        ax.set_title("Rolling Mean and Std Dev of Cosine Similarity")
        ax.set_xlabel("Batch Index")
        ax.set_ylabel("Cosine Similarity")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)

        ax = axs[0, 2]
        baseline_sims = group_vals[0]["cosine_scores"][:drift_start]
        drifted_sims = group_vals[0]["cosine_scores"][drift_start:]
        ax.hist(baseline_sims, bins=10, alpha=0.5, label="Baseline")
        ax.hist(drifted_sims, bins=10, alpha=0.5, label="Drifted")
        ax.set_title("Histogram of Similarities")
        ax.set_xlabel("Cosine Similarity")
        ax.set_ylabel("Frequency")
        ax.legend()

        ax = axs[1, 0]
        pca_group = [g for g in group_vals if g["pca"]]
        if pca_group:
            gv_pca = pca_group[0]
            baseline_embs = gv_pca.get("baseline_embs")
            drifted_embs = gv_pca.get("drifted_embs")
            if baseline_embs is not None and drifted_embs is not None:
                ax.scatter(
                    baseline_embs[:, 0],
                    baseline_embs[:, 1],
                    alpha=0.5,
                    label="Baseline",
                )
                ax.scatter(
                    drifted_embs[:, 0], drifted_embs[:, 1], alpha=0.5, label="Drifted"
                )
        ax.set_title("Scatter Plot of Embeddings (PCA Space)")
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.legend()

        ax = axs[1, 1]
        for gv in group_vals:
            baseline_similarity = gv["cosine_scores"][0]
            deltas = [sim - baseline_similarity for sim in gv["cosine_scores"]]
            label_suffix = "PCA" if gv["pca"] else "No PCA"
            ax.plot(deltas, label=f"{label_suffix}")
        ax.axhline(
            y=0, color="black", linestyle="--", alpha=0.5, label="Baseline Similarity"
        )
        ax.set_title("Delta from Baseline Similarity")
        ax.set_xlabel("Batch Index")
        ax.set_ylabel("Delta (Cosine Similarity)")
        ax.legend()

        ax = axs[1, 2]
        no_pca_vals = next(gv for gv in group_vals if not gv["pca"])
        pca_vals = next(gv for gv in group_vals if gv["pca"])
        ax.plot(no_pca_vals["cosine_scores"], label="No PCA", color="blue")
        ax.plot(pca_vals["cosine_scores"], label="PCA", color="orange")
        ax.set_title("No PCA vs PCA Cosine Similarity")
        ax.set_xlabel("Batch Index")
        ax.set_ylabel("Cosine Similarity")
        ax.legend()

        plt.suptitle(f"{dataset_name} | {model_name} | Drift={drift_strength}")
        plt.tight_layout()
        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_drift{drift_strength}_enhanced.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.close()
        print(f"  Saved enhanced plot: {save_path}")

    print("\nAll done!")


if __name__ == "__main__":
    main()

Using device: mps

=== Loading dataset: yelp_review_full ===

--- Using Model: nlpaueb/sec-bert-base ---

Simulating drift with strength=0.0


16it [00:06,  2.64it/s]
16it [00:06,  2.66it/s]


  Cosine scores (no PCA): 16 entries
  Cosine scores (PCA): 16 entries

Simulating drift with strength=0.5


16it [00:06,  2.65it/s]
16it [00:06,  2.65it/s]


  Cosine scores (no PCA): 16 entries
  Cosine scores (PCA): 16 entries

Simulating drift with strength=1.0


...

  Cosine scores (no PCA): 16 entries
  Cosine scores (PCA): 16 entries

Simulating drift with strength=0.5


16it [00:05,  2.74it/s]
16it [00:05,  2.73it/s]


  Cosine scores (no PCA): 16 entries
  Cosine scores (PCA): 16 entries

Simulating drift with strength=1.0


16it [00:05,  2.73it/s]
16it [00:05,  2.73it/s]


  Cosine scores (no PCA): 16 entries
  Cosine scores (PCA): 16 entries

=== Plotting results for each (dataset, model, drift_strength) ===
  Saved enhanced plot: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_drift0.0_enhanced.png
  Saved enhanced plot: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_drift0.5_enhanced.png
  Saved enhanced plot: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_drift1.0_enhanced.png
  Saved enhanced plot: results_multidataset/yelp_review_full_bert-base-uncased_drift0.0_enhanced.png
  Saved enhanced plot: results_multidataset/yelp_review_full_bert-base-uncased_drift0.5_enhanced.png
  Saved enhanced plot: results_multidataset/yelp_review_full_bert-base-uncased_drift1.0_enhanced.png
  Saved enhanced plot: results_multidataset/wikitext_nlpaueb_sec-bert-base_drift0.0_enhanced.png
  Saved enhanced plot: results_multidataset/wikitext_nlpaueb_sec-bert-base_drift0.5_enhanced.png
  Saved enhanced plot: results_multidataset/wikit